In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox, filedialog, scrolledtext
import pandas as pd
import numpy as np
from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
import threading
from datetime import datetime
import json
import os
import matplotlib.pyplot as plt
from matplotlib import cm
import warnings
import re
warnings.filterwarnings('ignore')

class CompressiveStrengthPredictor:
    def __init__(self):
        self.setup_style()
        self.setup_paths()
        self.load_data()
        self.train_test_split()
        self.train_model()
        self.create_gui()
        
    def setup_style(self):
        self.colors = {
            'primary': '#2E4053',        # Dark Slate
            'secondary': '#6A1E55',      # Deep Purple
            'accent': '#D4A373',          # Sandy Brown
            'background': '#F8F9FA',      # Light Gray
            'card_bg': '#FFFFFF',         # White
            'success': '#2D6A4F',         # Forest Green
            'warning': '#B9770E',          # Golden Brown
            'danger': '#B03A2E',           # Terracotta
            'sandy': '#D4A373',            # Sandy color
            'soil_brown': '#8B5A2B',       # Soil Brown
            'rf_green': '#2E8B57',         # Random Forest Green
            'pso_purple': '#800080',       # PSO Purple
            'strength_blue': '#2D6A4F',    # Strength blue
        }
        
    def setup_paths(self):
        # Updated path for your data
        self.data_path = r"D:\2026 Work\My Papers\2-Compressive Strength\DATA\Zero Normilisation.csv"
        # Updated save directory for plots
        self.save_dir = r"D:\2026 Work\My Papers\2-Compressive Strength\Permutation Feature importance"
        os.makedirs(self.save_dir, exist_ok=True)
        self.history_file = os.path.join(self.save_dir, "prediction_history.json")
        
    def clean_column_names(self, columns):
        """Clean column names by removing special characters"""
        cleaned = []
        for col in columns:
            # Replace special characters with underscore
            cleaned_col = re.sub(r'[^a-zA-Z0-9_ ]', '_', str(col))
            # Replace multiple spaces/underscores with single underscore
            cleaned_col = re.sub(r'[ _]+', '_', cleaned_col)
            # Remove leading/trailing underscores
            cleaned_col = cleaned_col.strip('_')
            # If empty after cleaning, use generic name
            if cleaned_col == '':
                cleaned_col = f'feature_{len(cleaned)}'
            cleaned.append(cleaned_col)
        return cleaned
        
    def load_data(self):
        try:
            print(f"📂 Loading data from: {self.data_path}")
            self.df = pd.read_csv(self.data_path, encoding='ISO-8859-1')
            print(f"✅ Dataset loaded successfully!")
            print(f"📐 Dataset shape: {self.df.shape}")
            
            # Clean column names - remove special characters
            self.df.columns = self.clean_column_names(self.df.columns)
            print(f"🔧 Cleaned columns: {self.df.columns.tolist()}")
            
            # Find target column - CS (Compressive Strength)
            target_col = None
            possible_targets = ['CS', 'Compressive_Strength', 'Compressive Strength', 
                               'CS_MPa', 'Strength', 'fc', 'fck']
            
            for target in possible_targets:
                # Try exact match first
                if target in self.df.columns:
                    target_col = target
                    break
                # Try case-insensitive match
                for col in self.df.columns:
                    if target.lower() in col.lower():
                        target_col = col
                        break
                if target_col:
                    break

            if target_col is None:
                # Use last column as fallback
                target_col = self.df.columns[-1]
                print(f"⚠️ Target column not found, using last column: '{target_col}'")
            else:
                print(f"✅ Target column found: '{target_col}'")
            
            self.target_name = target_col
            print(f"🎯 Target variable: '{self.target_name}'")
            
            # Prepare features and target
            self.X = self.df.drop(columns=[target_col])
            self.y = self.df[target_col]
            self.feature_names = self.X.columns.tolist()
            
            # Calculate statistics for each feature
            self.feature_stats = {}
            for feature in self.feature_names:
                self.feature_stats[feature] = {
                    'min': self.X[feature].min(),
                    'max': self.X[feature].max(),
                    'mean': self.X[feature].mean(),
                    'std': self.X[feature].std(),
                    'median': self.X[feature].median(),
                    'q1': self.X[feature].quantile(0.25),
                    'q3': self.X[feature].quantile(0.75)
                }
            
            print(f"🔍 Features loaded: {len(self.feature_names)}")
            print(f"📊 Features: {', '.join(self.feature_names[:10])}...")
            
            # Compressive strength statistics
            print(f"\n📊 Target Statistics - {self.target_name}:")
            print(f"   • Mean: {self.y.mean():.2f} MPa")
            print(f"   • Std: {self.y.std():.2f} MPa")
            print(f"   • Min: {self.y.min():.2f} MPa")
            print(f"   • Max: {self.y.max():.2f} MPa")
            
        except Exception as e:
            error_msg = f"Failed to load data: {str(e)}"
            print(f"❌ {error_msg}")
            messagebox.showerror("Data Loading Error", error_msg)
            raise

    def train_test_split(self):
        """Split data into training (70%) and testing (30%) sets"""
        try:
            print("\n" + "="*60)
            print("📊 TRAIN-TEST SPLIT (70-30)")
            print("="*60)
            
            # Split the data
            self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
                self.X, self.y, test_size=0.3, random_state=42
            )
            
            print(f"\n📈 Training set size: {len(self.X_train)} samples ({len(self.X_train)/len(self.X)*100:.1f}%)")
            print(f"📉 Testing set size: {len(self.X_test)} samples ({len(self.X_test)/len(self.X)*100:.1f}%)")
            
        except Exception as e:
            error_msg = f"Failed to split data: {str(e)}"
            print(f"❌ {error_msg}")
            messagebox.showerror("Data Split Error", error_msg)
            raise
    
    def train_model(self):
        try:
            print("\n" + "="*60)
            print("🚀 TRAINING RANDOM FOREST MODEL WITH PSO OPTIMIZED HYPERPARAMETERS")
            print("="*60)
            
            # Random Forest hyperparameters optimized by PSO
            self.hyperparams = {
                'n_estimators': 265,                 # Number of trees
                'max_depth': 10,                      # Maximum tree depth
                'min_samples_split': 2,               # Min samples to split
                'min_samples_leaf': 1,                 # Min samples per leaf
                'max_features': 0.565,                 # Feature fraction
                'max_samples': 0.888,                   # Sample fraction
                'min_impurity_decrease': 1e-05,        # Min impurity decrease
                'random_state': 42,
                'n_jobs': -1
            }
            
            print("\n📋 PSO-Optimized Hyperparameters:")
            for param, value in self.hyperparams.items():
                print(f"   • {param:25}: {value}")
            
            # Random Forest Regressor with PSO-optimized hyperparameters
            self.model = RandomForestRegressor(**self.hyperparams)
            
            print("\n🔄 Training on training set (70% of data)...")
            self.model.fit(self.X_train, self.y_train)
            print("✅ Random Forest model training completed!")
            
            # Make predictions on training and test sets
            self.y_train_pred = self.model.predict(self.X_train)
            self.y_test_pred = self.model.predict(self.X_test)
            
            # Calculate permutation feature importance on training data
            print("\n🔄 Calculating permutation feature importance...")
            perm_importance = permutation_importance(
                self.model, self.X_train, self.y_train, 
                n_repeats=10, random_state=42, n_jobs=-1
            )
            
            # Create feature importance dataframe
            self.feature_importance = pd.DataFrame({
                'feature': self.feature_names,
                'importance': perm_importance.importances_mean,
                'std': perm_importance.importances_std
            }).sort_values('importance', ascending=False)
            
            print(f"\n🏆 TOP 5 MOST IMPORTANT FEATURES (Permutation Importance):")
            for i, (_, row) in enumerate(self.feature_importance.head(5).iterrows(), 1):
                importance_pct = row['importance'] / self.feature_importance['importance'].sum() * 100
                bar = '█' * int(importance_pct/2) + '░' * (50 - int(importance_pct/2))
                print(f"   {i}. {row['feature'][:25]:25} : {row['importance']:.4f} ± {row['std']:.4f} |{bar}| {importance_pct:.1f}%")
            
            # Calculate metrics internally (not displayed)
            self._calculate_metrics_internal()
            
        except Exception as e:
            error_msg = f"Failed to train model: {str(e)}"
            print(f"❌ {error_msg}")
            messagebox.showerror("Model Training Error", error_msg)
            raise

    def _calculate_metrics_internal(self):
        """Calculate model performance metrics internally (not displayed)"""
        # Training set metrics
        ss_res_train = np.sum((self.y_train - self.y_train_pred) ** 2)
        ss_tot_train = np.sum((self.y_train - np.mean(self.y_train)) ** 2)
        self.r2_train = 1 - (ss_res_train / ss_tot_train)
        self.rmse_train = np.sqrt(np.mean((self.y_train_pred - self.y_train) ** 2))
        self.mae_train = np.mean(np.abs(self.y_train_pred - self.y_train))
        
        # Test set metrics
        ss_res_test = np.sum((self.y_test - self.y_test_pred) ** 2)
        ss_tot_test = np.sum((self.y_test - np.mean(self.y_test)) ** 2)
        self.r2_test = 1 - (ss_res_test / ss_tot_test)
        self.rmse_test = np.sqrt(np.mean((self.y_test_pred - self.y_test) ** 2))
        self.mae_test = np.mean(np.abs(self.y_test_pred - self.y_test))
        
        # MAPE for test set (avoid division by zero)
        mask = self.y_test != 0
        if mask.any():
            self.mape_test = np.mean(np.abs((self.y_test[mask] - self.y_test_pred[mask]) / self.y_test[mask])) * 100
        else:
            self.mape_test = np.nan
    
    def create_gui(self):
        self.root = tk.Tk()
        self.root.title("🏗️ Compressive Strength Predictor - Random Forest (PSO-Optimized)")
        self.root.geometry("1500x950")
        self.root.configure(bg=self.colors['background'])
        
        # Center window
        self.root.update_idletasks()
        width = self.root.winfo_width()
        height = self.root.winfo_height()
        x = (self.root.winfo_screenwidth() // 2) - (width // 2)
        y = (self.root.winfo_screenheight() // 2) - (height // 2)
        self.root.geometry(f'1500x950+{x}+{y}')
        
        # Configure ttk styles
        self.setup_styles()
        
        # Create notebook for tabs
        self.notebook = ttk.Notebook(self.root)
        self.notebook.pack(fill='both', expand=True, padx=10, pady=10)
        
        # Create tabs
        self.prediction_tab = self.create_prediction_tab()
        self.analysis_tab = self.create_analysis_tab()
        self.history_tab = self.create_history_tab()
        self.model_info_tab = self.create_model_info_tab()
        
        # Add tabs to notebook
        self.notebook.add(self.prediction_tab, text="🔮 Prediction")
        self.notebook.add(self.analysis_tab, text="📊 Analysis")
        self.notebook.add(self.history_tab, text="📋 History")
        self.notebook.add(self.model_info_tab, text="ℹ️ Model Info")
        
        # Initialize prediction history
        self.prediction_history = []
        self.load_history()
        
        print("\n✅ GUI created successfully!")
        print("🏁 Application ready for predictions\n")
    
    def setup_styles(self):
        style = ttk.Style()
        style.theme_use('clam')
        
        # Configure custom styles
        style.configure('Accent.TButton',
                       background=self.colors['accent'],
                       foreground='white',
                       font=('Arial', 11, 'bold'),
                       borderwidth=2,
                       relief='raised')
        
        style.map('Accent.TButton',
                 background=[('active', '#C19A6B'), ('pressed', '#A67B5B')])
        
        style.configure('Success.TButton',
                       background=self.colors['success'],
                       foreground='white',
                       font=('Arial', 11, 'bold'),
                       borderwidth=2)
        
        style.map('Success.TButton',
                 background=[('active', '#1E4D3A'), ('pressed', '#163A2B')])
        
        style.configure('Primary.TButton',
                       background=self.colors['primary'],
                       foreground='white',
                       font=('Arial', 10, 'bold'))
        
        style.configure('Soil.TLabel',
                       background=self.colors['sandy'],
                       foreground='white',
                       font=('Arial', 10, 'bold'))
        
        style.configure('Header.TLabel',
                       font=('Arial', 14, 'bold'),
                       foreground=self.colors['primary'])
        
        style.configure('SubHeader.TLabel',
                       font=('Arial', 11, 'bold'),
                       foreground=self.colors['soil_brown'])
        
        style.configure('Card.TFrame',
                       background=self.colors['card_bg'],
                       relief='raised',
                       borderwidth=2)
    
    def create_prediction_tab(self):
        tab = ttk.Frame(self.notebook)
        
        # Main container with two panels
        main_paned = ttk.PanedWindow(tab, orient=tk.HORIZONTAL)
        main_paned.pack(fill='both', expand=True, padx=10, pady=10)
        
        # Left panel - Inputs
        left_panel = ttk.Frame(main_paned)
        main_paned.add(left_panel, weight=40)
        
        # Right panel - Results
        right_panel = ttk.Frame(main_paned)
        main_paned.add(right_panel, weight=60)
        
        # ========== LEFT PANEL: Input Parameters ==========
        
        # Header with concrete icon
        header_frame = ttk.Frame(left_panel)
        header_frame.pack(fill='x', pady=(0, 10))
        
        header_label = tk.Label(header_frame,
                               text="🏗️ COMPRESSIVE STRENGTH - MIX PARAMETERS",
                               font=("Arial", 16, "bold"),
                               fg=self.colors['soil_brown'])
        header_label.pack()
        
        subheader = tk.Label(header_frame,
                            text="Enter mix parameters to predict compressive strength",
                            font=("Arial", 10),
                            fg=self.colors['primary'])
        subheader.pack()
        
        # Quick selection frame
        quick_frame = ttk.LabelFrame(left_panel, text="⚡ Quick Selections", padding=15)
        quick_frame.pack(fill='x', pady=(0, 15))
        
        # Model info with styled display
        model_info_frame = ttk.Frame(quick_frame)
        model_info_frame.pack(fill='x', pady=(0, 10))
        
        model_badge = tk.Label(model_info_frame,
                              text="Random Forest + PSO",
                              font=("Arial", 10, "bold"),
                              bg=self.colors['rf_green'],
                              fg='white',
                              padx=10,
                              pady=5)
        model_badge.pack(side='left', padx=5)
        
        importance_badge = tk.Label(model_info_frame,
                                   text="Permutation Importance",
                                   font=("Arial", 10, "bold"),
                                   bg=self.colors['pso_purple'],
                                   fg='white',
                                   padx=10,
                                   pady=5)
        importance_badge.pack(side='left', padx=5)
        
        split_badge = tk.Label(model_info_frame,
                              text="70-30 Split",
                              font=("Arial", 10, "bold"),
                              bg=self.colors['primary'],
                              fg='white',
                              padx=10,
                              pady=5)
        split_badge.pack(side='left', padx=5)
        
        features_badge = tk.Label(model_info_frame,
                                 text=f"Features: {len(self.feature_names)}",
                                 font=("Arial", 10, "bold"),
                                 bg=self.colors['secondary'],
                                 fg='white',
                                 padx=10,
                                 pady=5)
        features_badge.pack(side='left', padx=5)
        
        # Preset selection
        ttk.Label(quick_frame, text="Mix Design Preset:", 
                 font=("Arial", 10, "bold")).pack(anchor='w', pady=(5, 5))
        
        self.preset_var = tk.StringVar(value="Typical Mix")
        preset_combo = ttk.Combobox(quick_frame, textvariable=self.preset_var, 
                                   values=["Low Strength Mix", "Medium Strength Mix", 
                                           "High Strength Mix", "Very High Strength Mix",
                                           "Custom"],
                                   state="readonly", width=25, font=("Arial", 10))
        preset_combo.pack(fill='x', pady=(0, 10))
        preset_combo.bind("<<ComboboxSelected>>", self.load_preset)
        
        # Quick action buttons
        action_frame = ttk.Frame(quick_frame)
        action_frame.pack(fill='x', pady=5)
        
        buttons = [
            ("🎲 Random", self.fill_random_sample, 'Primary.TButton'),
            ("📊 Min Strength", self.fill_min_strength, 'Primary.TButton'),
            ("📈 Max Strength", self.fill_max_strength, 'Primary.TButton'),
            ("🔄 Clear", self.clear_inputs, 'Primary.TButton')
        ]
        
        for text, cmd, style in buttons:
            btn = ttk.Button(action_frame, text=text, command=cmd, style=style)
            btn.pack(side='left', padx=2, expand=True, fill='x')
        
        # Feature inputs frame with scrollbar
        input_container = ttk.LabelFrame(left_panel, text="📊 Mix Parameters", padding=15)
        input_container.pack(fill='both', expand=True)
        
        # Create canvas with scrollbar
        canvas = tk.Canvas(input_container, highlightthickness=0)
        scrollbar = ttk.Scrollbar(input_container, orient="vertical", command=canvas.yview)
        scrollable_frame = ttk.Frame(canvas)
        
        scrollable_frame.bind(
            "<Configure>",
            lambda e: canvas.configure(scrollregion=canvas.bbox("all"))
        )
        
        canvas.create_window((0, 0), window=scrollable_frame, anchor="nw")
        canvas.configure(yscrollcommand=scrollbar.set)
        
        canvas.pack(side="left", fill="both", expand=True)
        scrollbar.pack(side="right", fill="y")
        
        # Bind mousewheel for scrolling
        def _on_mousewheel(event):
            canvas.yview_scroll(int(-1*(event.delta/120)), "units")
        
        canvas.bind_all("<MouseWheel>", _on_mousewheel)
        
        # Feature entries
        self.entries = {}
        
        # Show all features with importance indicators
        for feature in self.feature_names:
            feat_frame = ttk.Frame(scrollable_frame)
            feat_frame.pack(fill='x', pady=8)
            
            stats = self.feature_stats[feature]
            importance_info = self.feature_importance[self.feature_importance['feature'] == feature]
            
            if not importance_info.empty:
                importance = importance_info['importance'].values[0]
                importance_std = importance_info['std'].values[0]
                importance_pct = importance / self.feature_importance['importance'].sum() * 100
            else:
                importance = 0
                importance_std = 0
                importance_pct = 0
            
            # Feature name with importance bar
            name_frame = ttk.Frame(feat_frame)
            name_frame.pack(fill='x')
            
            label = tk.Label(name_frame,
                            text=f"◉ {feature}",
                            font=("Arial", 9, "bold"),
                            fg=self.get_importance_color(importance),
                            anchor='w')
            label.pack(side='left')
            
            # Importance mini bar
            bar_width = int(importance_pct / 2)
            bar = '█' * bar_width + '░' * (20 - bar_width)
            imp_label = tk.Label(name_frame,
                                text=f"{bar} {importance_pct:.1f}%",
                                font=("Arial", 8),
                                fg=self.colors['primary'])
            imp_label.pack(side='right')
            
            # Input row
            input_row = ttk.Frame(feat_frame)
            input_row.pack(fill='x', pady=(2, 0))
            
            # Entry field
            entry = ttk.Entry(input_row, width=15, font=("Arial", 10))
            entry.insert(0, f"{stats['mean']:.3f}")
            entry.pack(side='left', padx=(0, 5))
            
            # Range label
            range_label = tk.Label(input_row,
                                  text=f"[{stats['min']:.2f} - {stats['max']:.2f}]",
                                  font=("Arial", 8),
                                  fg=self.colors['primary'])
            range_label.pack(side='left', padx=5)
            
            # Units based on feature name
            if 'cement' in feature.lower():
                unit = "kg/m³"
            elif 'water' in feature.lower():
                unit = "kg/m³"
            elif 'aggregate' in feature.lower() or 'sand' in feature.lower():
                unit = "kg/m³"
            elif 'admixture' in feature.lower():
                unit = "kg/m³"
            elif 'ratio' in feature.lower():
                unit = ""
            elif 'age' in feature.lower():
                unit = "days"
            else:
                unit = ""
            
            if unit:
                unit_label = tk.Label(input_row,
                                     text=unit,
                                     font=("Arial", 8, "bold"),
                                     fg=self.colors['soil_brown'])
                unit_label.pack(side='left')
            
            self.entries[feature] = entry
            
            # Slider
            if stats['max'] - stats['min'] > 0:
                slider = ttk.Scale(input_row,
                                  from_=stats['min'],
                                  to=stats['max'],
                                  value=stats['mean'],
                                  orient='horizontal',
                                  length=150,
                                  command=lambda v, f=feature: self.update_entry_from_slider(f, float(v)))
                slider.pack(side='right', padx=5)
        
        # Predict button
        predict_frame = ttk.Frame(left_panel)
        predict_frame.pack(fill='x', pady=15)
        
        self.predict_btn = ttk.Button(predict_frame,
                                     text="🔮 PREDICT COMPRESSIVE STRENGTH",
                                     command=self.threaded_predict,
                                     style='Accent.TButton')
        self.predict_btn.pack(fill='x', ipady=10)
        
        # ========== RIGHT PANEL: Results ==========
        
        # Header
        result_header = tk.Label(right_panel,
                                text="PREDICTION RESULTS",
                                font=("Arial", 16, "bold"),
                                fg=self.colors['soil_brown'])
        result_header.pack(pady=(10, 15))
        
        # Main result display
        result_card = ttk.Frame(right_panel, style='Card.TFrame')
        result_card.pack(fill='x', padx=10, pady=10)
        
        self.result_var = tk.StringVar(value="---")
        result_label = tk.Label(result_card,
                               textvariable=self.result_var,
                               font=("Arial", 48, "bold"),
                               fg=self.colors['strength_blue'])
        result_label.pack(pady=20)
        
        unit_label = tk.Label(result_card,
                             text="Compressive Strength (MPa)",
                             font=("Arial", 14),
                             fg=self.colors['primary'])
        unit_label.pack()
        
        # Classification only (confidence removed)
        info_card = ttk.Frame(right_panel, style='Card.TFrame')
        info_card.pack(fill='x', padx=10, pady=10)
        
        classification_frame = ttk.Frame(info_card)
        classification_frame.pack(fill='x', pady=20, padx=10)
        
        tk.Label(classification_frame, text="Strength Classification:",
                font=("Arial", 14, "bold")).pack(anchor='center')
        
        self.classification_var = tk.StringVar(value="Not predicted")
        self.classification_label = tk.Label(classification_frame,
                                            textvariable=self.classification_var,
                                            font=("Arial", 24, "bold"))
        self.classification_label.pack(pady=10)
        
        # Detailed results notebook
        detail_notebook = ttk.Notebook(right_panel)
        detail_notebook.pack(fill='both', expand=True, padx=10, pady=10)
        
        # Tab 1: Feature Contributions
        contributions_tab = ttk.Frame(detail_notebook)
        detail_notebook.add(contributions_tab, text="📊 Feature Contributions")
        
        self.contributions_text = scrolledtext.ScrolledText(contributions_tab,
                                                           font=("Arial", 10),
                                                           wrap=tk.WORD,
                                                           height=12)
        self.contributions_text.pack(fill='both', expand=True, padx=5, pady=5)
        self.contributions_text.insert(1.0, "Feature contributions will appear here after prediction.\n\n"
                                            "Top features and their impact on compressive strength:")
        self.contributions_text.config(state='disabled')
        
        # Tab 2: Comparison
        comparison_tab = ttk.Frame(detail_notebook)
        detail_notebook.add(comparison_tab, text="📈 Comparison")
        
        self.comparison_text = scrolledtext.ScrolledText(comparison_tab,
                                                        font=("Arial", 10),
                                                        wrap=tk.WORD,
                                                        height=12)
        self.comparison_text.pack(fill='both', expand=True, padx=5, pady=5)
        self.comparison_text.insert(1.0, "Comparison with typical values will appear here.")
        self.comparison_text.config(state='disabled')
        
        # Action buttons
        action_frame = ttk.Frame(right_panel)
        action_frame.pack(fill='x', padx=10, pady=10)
        
        action_buttons = [
            ("💾 Save Result", self.save_result),
            ("📋 Copy", self.copy_to_clipboard),
            ("📤 Export", self.export_results),
            ("📊 Analyze", lambda: self.notebook.select(1))
        ]
        
        for text, cmd in action_buttons:
            btn = ttk.Button(action_frame, text=text, command=cmd, style='Primary.TButton')
            btn.pack(side='left', padx=2, expand=True, fill='x')
        
        return tab
    
    def create_analysis_tab(self):
        tab = ttk.Frame(self.notebook)
        
        # Analysis controls
        control_frame = ttk.LabelFrame(tab, text="📊 Analysis Tools", padding=20)
        control_frame.pack(fill='x', padx=20, pady=10)
        
        # Analysis buttons grid
        button_grid = ttk.Frame(control_frame)
        button_grid.pack()
        
        analysis_buttons = [
            ("🏆 Permutation Importance", self.plot_feature_importance, self.colors['pso_purple']),
            ("📊 Data Distribution", self.plot_data_distribution, self.colors['success']),
            ("🔗 Correlation Matrix", self.plot_correlation_matrix, self.colors['danger']),
            ("📋 Feature Statistics", self.plot_feature_statistics, self.colors['rf_green']),
            ("🏗️ Mix Parameters", self.plot_mix_parameters, self.colors['sandy']),
        ]
        
        for i, (text, command, color) in enumerate(analysis_buttons):
            row, col = divmod(i, 3)
            btn = tk.Button(button_grid, text=text, command=command,
                          bg=color, fg='white', font=("Arial", 10, "bold"),
                          padx=15, pady=8, relief='raised', bd=2,
                          cursor='hand2', width=18)
            btn.grid(row=row, column=col, padx=5, pady=5)
            
            # Hover effect
            btn.bind("<Enter>", lambda e, b=btn: b.config(bg=self.lighten_color(color)))
            btn.bind("<Leave>", lambda e, b=btn, c=color: b.config(bg=c))
        
        # Save plots button
        save_frame = ttk.Frame(control_frame)
        save_frame.pack(pady=(15, 0))
        
        tk.Button(save_frame, text="💾 SAVE ALL PLOTS TO FOLDER",
                 command=self.save_all_plots,
                 bg=self.colors['soil_brown'], fg='white',
                 font=("Arial", 12, "bold"),
                 padx=30, pady=10, relief='raised', bd=3,
                 cursor='hand2').pack()
        
        # Plot area
        self.analysis_frame = ttk.Frame(tab)
        self.analysis_frame.pack(fill='both', expand=True, padx=20, pady=10)
        
        # Initial plot
        self.plot_feature_importance()
        
        return tab
    
    def create_history_tab(self):
        tab = ttk.Frame(self.notebook)
        
        # Header
        header_frame = ttk.Frame(tab)
        header_frame.pack(fill='x', padx=20, pady=10)
        
        tk.Label(header_frame, text="📋 PREDICTION HISTORY",
                font=("Arial", 18, "bold"),
                fg=self.colors['soil_brown']).pack()
        
        # History controls
        control_frame = ttk.LabelFrame(tab, text="History Management", padding=15)
        control_frame.pack(fill='x', padx=20, pady=10)
        
        button_frame = ttk.Frame(control_frame)
        button_frame.pack()
        
        history_buttons = [
            ("🗑️ Clear History", self.clear_history, self.colors['danger']),
            ("💾 Export to CSV", self.export_history, self.colors['success']),
            ("📤 Load History", self.load_history_from_file, self.colors['primary']),
            ("🔄 Refresh", self.update_history_display, self.colors['pso_purple']),
        ]
        
        for text, cmd, color in history_buttons:
            btn = tk.Button(button_frame, text=text, command=cmd,
                          bg=color, fg='white', font=("Arial", 10, "bold"),
                          padx=10, pady=5, relief='raised', bd=2,
                          cursor='hand2')
            btn.pack(side='left', padx=5)
        
        # Statistics
        self.history_stats_var = tk.StringVar(value="No predictions yet")
        stats_label = tk.Label(control_frame,
                              textvariable=self.history_stats_var,
                              font=("Arial", 12, "bold"),
                              fg=self.colors['strength_blue'])
        stats_label.pack(pady=(10, 0))
        
        # History display
        display_frame = ttk.Frame(tab)
        display_frame.pack(fill='both', expand=True, padx=20, pady=(0, 20))
        
        # Create treeview for history - Confidence column removed
        columns = ('Timestamp', 'Strength (MPa)', 'Classification', 'Top Feature')
        self.history_tree = ttk.Treeview(display_frame, columns=columns, show='headings', height=20)
        
        # Define headings
        self.history_tree.heading('Timestamp', text='Date & Time')
        self.history_tree.heading('Strength (MPa)', text='Strength (MPa)')
        self.history_tree.heading('Classification', text='Classification')
        self.history_tree.heading('Top Feature', text='Top Feature')
        
        # Define column widths
        self.history_tree.column('Timestamp', width=180)
        self.history_tree.column('Strength (MPa)', width=120)
        self.history_tree.column('Classification', width=180)
        self.history_tree.column('Top Feature', width=200)
        
        # Add scrollbars
        vsb = ttk.Scrollbar(display_frame, orient="vertical", command=self.history_tree.yview)
        hsb = ttk.Scrollbar(display_frame, orient="horizontal", command=self.history_tree.xview)
        self.history_tree.configure(yscrollcommand=vsb.set, xscrollcommand=hsb.set)
        
        # Grid layout
        self.history_tree.grid(row=0, column=0, sticky='nsew')
        vsb.grid(row=0, column=1, sticky='ns')
        hsb.grid(row=1, column=0, sticky='ew')
        
        display_frame.grid_rowconfigure(0, weight=1)
        display_frame.grid_columnconfigure(0, weight=1)
        
        # Bind double-click
        self.history_tree.bind('<Double-Button-1>', self.load_history_entry)
        
        return tab
    
    def create_model_info_tab(self):
        tab = ttk.Frame(self.notebook)
        
        # Main container with scrollbar
        canvas = tk.Canvas(tab)
        scrollbar = ttk.Scrollbar(tab, orient="vertical", command=canvas.yview)
        scrollable_frame = ttk.Frame(canvas)
        
        scrollable_frame.bind(
            "<Configure>",
            lambda e: canvas.configure(scrollregion=canvas.bbox("all"))
        )
        
        canvas.create_window((0, 0), window=scrollable_frame, anchor="nw")
        canvas.configure(yscrollcommand=scrollbar.set)
        
        canvas.pack(side="left", fill="both", expand=True)
        scrollbar.pack(side="right", fill="y")
        
        # Model Information
        info_frame = ttk.LabelFrame(scrollable_frame, text="🚀 Random Forest + PSO Model Information", padding=20)
        info_frame.pack(fill='x', padx=20, pady=10)
        
        # Model overview - NO STATISTICAL PARAMETERS SHOWN
        overview_text = f"""
═══════════════════════════════════════════════════════════════
   COMPRESSIVE STRENGTH PREDICTOR - RANDOM FOREST + PSO
═══════════════════════════════════════════════════════════════

🎯 Target Variable: {self.target_name}
📊 Number of Features: {len(self.feature_names)}
📈 Training Samples: {len(self.X_train)} (70%)
📉 Testing Samples: {len(self.X_test)} (30%)
🚀 Algorithm: Random Forest Regressor (PSO-Optimized)
📊 Importance: Permutation Feature Importance

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
         PSO-OPTIMIZED HYPERPARAMETERS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""
        
        for param, value in self.hyperparams.items():
            overview_text += f"• {param:25}: {value}\n"
        
        model_info_label = tk.Label(info_frame, text=overview_text,
                                   font=("Courier", 10),
                                   justify='left',
                                   bg='white',
                                   padx=20, pady=20)
        model_info_label.pack(fill='x')
        
        # Feature Importance Table
        importance_frame = ttk.LabelFrame(scrollable_frame, text="📊 Permutation Feature Importance", padding=20)
        importance_frame.pack(fill='x', padx=20, pady=10)
        
        # Create treeview for feature importance
        columns = ('Rank', 'Feature', 'Importance', 'Std Dev', 'Percentage')
        importance_tree = ttk.Treeview(importance_frame, columns=columns, show='headings', height=15)
        
        importance_tree.heading('Rank', text='Rank')
        importance_tree.heading('Feature', text='Feature Name')
        importance_tree.heading('Importance', text='Importance Score')
        importance_tree.heading('Std Dev', text='Std Deviation')
        importance_tree.heading('Percentage', text='Percentage (%)')
        
        importance_tree.column('Rank', width=50)
        importance_tree.column('Feature', width=200)
        importance_tree.column('Importance', width=120)
        importance_tree.column('Std Dev', width=100)
        importance_tree.column('Percentage', width=100)
        
        # Add scrollbar
        tree_scroll = ttk.Scrollbar(importance_frame, orient="vertical", command=importance_tree.yview)
        importance_tree.configure(yscrollcommand=tree_scroll.set)
        
        importance_tree.pack(side='left', fill='both', expand=True)
        tree_scroll.pack(side='right', fill='y')
        
        # Add data
        total_importance = self.feature_importance['importance'].sum()
        
        for i, (_, row) in enumerate(self.feature_importance.iterrows(), 1):
            percentage = (row['importance'] / total_importance) * 100
            importance_tree.insert('', 'end', values=(
                i,
                row['feature'][:30],
                f"{row['importance']:.6f}",
                f"{row['std']:.6f}",
                f"{percentage:.2f}%"
            ))
        
        # Data Statistics
        stats_frame = ttk.LabelFrame(scrollable_frame, text="📈 Data Statistics", padding=20)
        stats_frame.pack(fill='x', padx=20, pady=10)
        
        # Create two columns
        stats_columns = ttk.Frame(stats_frame)
        stats_columns.pack(fill='x')
        
        left_stats = ttk.Frame(stats_columns)
        left_stats.pack(side='left', fill='both', expand=True)
        
        right_stats = ttk.Frame(stats_columns)
        right_stats.pack(side='right', fill='both', expand=True)
        
        # Target statistics
        target_stats = f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━
    TARGET STATISTICS
━━━━━━━━━━━━━━━━━━━━━━━━━━
Variable: {self.target_name}
Count: {len(self.y)}
Mean: {self.y.mean():.2f} MPa
Std: {self.y.std():.2f} MPa
Min: {self.y.min():.2f} MPa
25%: {self.y.quantile(0.25):.2f} MPa
50%: {self.y.median():.2f} MPa
75%: {self.y.quantile(0.75):.2f} MPa
Max: {self.y.max():.2f} MPa
"""
        
        target_label = tk.Label(left_stats, text=target_stats,
                              font=("Courier", 10),
                              justify='left',
                              bg='#f8f9fa',
                              padx=10, pady=10)
        target_label.pack(fill='x')
        
        # Feature statistics summary
        feature_stats_summary = f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━
    FEATURES SUMMARY
━━━━━━━━━━━━━━━━━━━━━━━━━━
Number of Features: {len(self.feature_names)}

Top 5 Features (Permutation Importance):
"""
        
        for i, (_, row) in enumerate(self.feature_importance.head(5).iterrows(), 1):
            feature_stats_summary += f"{i}. {row['feature'][:20]}: {row['importance']:.6f} ± {row['std']:.6f}\n"
        
        feature_label = tk.Label(right_stats, text=feature_stats_summary,
                               font=("Courier", 10),
                               justify='left',
                               bg='#f8f9fa',
                               padx=10, pady=10)
        feature_label.pack(fill='x')
        
        # Strength Classification Guide
        guide_frame = ttk.LabelFrame(scrollable_frame, text="📋 Strength Classification Guide", padding=20)
        guide_frame.pack(fill='x', padx=20, pady=10)
        
        guide_text = """
═══════════════════════════════════════════════════════════════
   COMPRESSIVE STRENGTH CLASSIFICATION
═══════════════════════════════════════════════════════════════

Based on the predicted compressive strength (MPa):

🟢 VERY LOW STRENGTH   : < 20 MPa - Suitable for non-structural applications
🟡 LOW STRENGTH        : 20-30 MPa - General construction, slabs, foundations
🟠 MEDIUM STRENGTH     : 30-40 MPa - Structural elements, beams, columns
🔴 HIGH STRENGTH       : 40-50 MPa - High-rise buildings, bridges
🔥 VERY HIGH STRENGTH  : > 50 MPa - Special structures, pre-stressed concrete

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        FACTORS AFFECTING COMPRESSIVE STRENGTH
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

• Water-Cement Ratio : Lower ratio → Higher strength
• Cement Content : Higher cement content → Higher strength
• Aggregate Properties : Size, shape, and grading affect strength
• Curing Conditions : Proper curing essential for strength development
• Admixtures : Can enhance or modify strength properties
"""
        
        guide_label = tk.Label(guide_frame, text=guide_text,
                             font=("Courier", 10),
                             justify='left',
                             bg='white',
                             padx=20, pady=20)
        guide_label.pack(fill='x')
        
        return tab
    
    def get_importance_color(self, importance):
        if importance > 0.1:
            return self.colors['danger']
        elif importance > 0.05:
            return self.colors['warning']
        elif importance > 0.02:
            return self.colors['success']
        else:
            return self.colors['primary']
    
    def lighten_color(self, color):
        """Lighten a color for hover effect"""
        lightened = {
            self.colors['primary']: '#3A506B',
            self.colors['secondary']: '#883C6B',
            self.colors['success']: '#3C8D6B',
            self.colors['warning']: '#D49C3A',
            self.colors['danger']: '#C44B3E',
            self.colors['sandy']: '#E6B88A',
            self.colors['pso_purple']: '#9B30FF',
            self.colors['rf_green']: '#3CB371',
            self.colors['soil_brown']: '#A67B5B',
        }
        return lightened.get(color, color)
    
    def load_preset(self, event=None):
        preset = self.preset_var.get()
        self.load_preset_by_name(preset)
    
    def load_preset_by_name(self, preset_name):
        # Define presets based on concrete strength characteristics
        presets = {
            "Low Strength Mix": {
                "description": "Low strength concrete (< 20 MPa) for non-structural applications",
                "multiplier": 0.7
            },
            "Medium Strength Mix": {
                "description": "Medium strength concrete (20-30 MPa) for general construction",
                "multiplier": 1.0
            },
            "High Strength Mix": {
                "description": "High strength concrete (30-40 MPa) for structural elements",
                "multiplier": 1.3
            },
            "Very High Strength Mix": {
                "description": "Very high strength concrete (> 40 MPa) for special structures",
                "multiplier": 1.6
            }
        }
        
        if preset_name in presets:
            multiplier = presets[preset_name]["multiplier"]
            
            for feature in self.entries:
                stats = self.feature_stats[feature]
                
                # Adjust based on feature name
                if 'cement' in feature.lower():
                    adj_multiplier = multiplier * 1.2
                elif 'water' in feature.lower():
                    adj_multiplier = 1.0 / max(multiplier, 0.5)  # Inverse for water
                elif 'aggregate' in feature.lower() or 'sand' in feature.lower():
                    adj_multiplier = multiplier * 1.1
                else:
                    adj_multiplier = multiplier
                
                new_value = stats['mean'] * adj_multiplier
                
                # Ensure within bounds
                new_value = max(stats['min'], min(stats['max'], new_value))
                
                self.entries[feature].delete(0, tk.END)
                self.entries[feature].insert(0, f"{new_value:.3f}")
            
            messagebox.showinfo("Preset Loaded", 
                               f"✅ Loaded '{preset_name}' preset\n📝 {presets[preset_name]['description']}")
    
    def fill_random_sample(self):
        """Fill with a random sample from dataset"""
        random_idx = np.random.randint(0, len(self.df))
        sample = self.df.iloc[random_idx]
        
        for feature in self.entries:
            if feature in sample:
                self.entries[feature].delete(0, tk.END)
                value = sample[feature]
                self.entries[feature].insert(0, f"{value:.3f}")
        
        actual_strength = sample[self.target_name]
        messagebox.showinfo("Random Sample", 
                           f"✅ Loaded random sample #{random_idx}\n📊 Actual strength: {actual_strength:.2f} MPa")
    
    def fill_min_strength(self):
        """Fill with values that give minimum compressive strength"""
        best_idx = self.y.idxmin()
        sample = self.df.iloc[best_idx]
        
        for feature in self.entries:
            if feature in sample:
                self.entries[feature].delete(0, tk.END)
                self.entries[feature].insert(0, f"{sample[feature]:.3f}")
        
        messagebox.showinfo("Minimum Strength", 
                           f"✅ Loaded minimum strength scenario\n📊 Min strength in dataset: {self.y.min():.2f} MPa")
    
    def fill_max_strength(self):
        """Fill with values that give maximum compressive strength"""
        worst_idx = self.y.idxmax()
        sample = self.df.iloc[worst_idx]
        
        for feature in self.entries:
            if feature in sample:
                self.entries[feature].delete(0, tk.END)
                self.entries[feature].insert(0, f"{sample[feature]:.3f}")
        
        messagebox.showinfo("Maximum Strength", 
                           f"✅ Loaded maximum strength scenario\n📊 Max strength in dataset: {self.y.max():.2f} MPa")
    
    def update_entry_from_slider(self, feature, value):
        """Update entry field when slider moves"""
        if feature in self.entries:
            self.entries[feature].delete(0, tk.END)
            self.entries[feature].insert(0, f"{value:.3f}")
    
    def threaded_predict(self):
        """Run prediction in separate thread"""
        self.predict_btn.config(state='disabled', text="⏳ Predicting...")
        thread = threading.Thread(target=self.predict_strength)
        thread.daemon = True
        thread.start()
    
    def predict_strength(self):
        try:
            # Get input values
            inputs = {}
            for feature, entry in self.entries.items():
                value = entry.get().strip()
                if value:
                    try:
                        inputs[feature] = float(value)
                    except:
                        inputs[feature] = self.feature_stats[feature]['mean']
                else:
                    inputs[feature] = self.feature_stats[feature]['mean']
            
            # Fill missing features with their means
            for feature in self.feature_names:
                if feature not in inputs:
                    inputs[feature] = self.feature_stats[feature]['mean']
            
            # Create input data
            input_data = pd.DataFrame([inputs], columns=self.feature_names)
            
            # Make prediction
            prediction = self.model.predict(input_data)[0]
            
            # Calculate feature contributions (using permutation importance as proxy)
            feature_contributions = {}
            predictions_with_variation = []
            
            for feature in self.feature_importance.head(5)['feature']:
                if feature in inputs:
                    # Create variation
                    temp_inputs = inputs.copy()
                    stats = self.feature_stats[feature]
                    temp_inputs[feature] = stats['mean']
                    temp_data = pd.DataFrame([temp_inputs], columns=self.feature_names)
                    base_pred = self.model.predict(temp_data)[0]
                    
                    importance = self.feature_importance[self.feature_importance['feature'] == feature]['importance'].values[0]
                    contribution = (prediction - base_pred) * importance / 100
                    feature_contributions[feature] = contribution
                    
                    # For comparison
                    predictions_with_variation.append({
                        'feature': feature,
                        'value': inputs[feature],
                        'mean': stats['mean'],
                        'prediction': prediction
                    })
            
            # Update GUI - confidence removed
            self.root.after(0, lambda: self.display_result(
                prediction, inputs, feature_contributions, predictions_with_variation))
            
        except Exception as e:
            self.root.after(0, lambda: messagebox.showerror(
                "Prediction Error", f"❌ Prediction failed: {str(e)}"))
        
        self.root.after(0, self.enable_predict_button)
    
    def enable_predict_button(self):
        self.predict_btn.config(state='normal', text="🔮 PREDICT COMPRESSIVE STRENGTH")
    
    def display_result(self, prediction, inputs, feature_contributions, comparisons):
        # Format result
        result_text = f"{prediction:.2f}"
        self.result_var.set(result_text)
        
        # Determine classification
        classification, color = self.classify_strength(prediction)
        self.classification_var.set(classification)
        self.classification_label.config(fg=color)
        
        # Update feature contributions
        self.update_contributions_text(feature_contributions, prediction)
        
        # Update comparison text
        self.update_comparison_text(comparisons, prediction)
        
        # Save to history (confidence removed)
        self.save_to_history(prediction, classification, inputs)
        
        # Success message
        self.show_success_animation()
    
    def classify_strength(self, strength_value):
        if strength_value >= 50:
            return "🔥 VERY HIGH STRENGTH", self.colors['danger']
        elif strength_value >= 40:
            return "🔴 HIGH STRENGTH", self.colors['warning']
        elif strength_value >= 30:
            return "🟠 MEDIUM STRENGTH", self.colors['accent']
        elif strength_value >= 20:
            return "🟡 LOW STRENGTH", self.colors['success']
        else:
            return "🟢 VERY LOW STRENGTH", self.colors['rf_green']
    
    def update_contributions_text(self, contributions, prediction):
        self.contributions_text.config(state='normal')
        self.contributions_text.delete(1.0, tk.END)
        
        if contributions:
            text = f"📊 FEATURE CONTRIBUTIONS: {prediction:.2f} MPa\n"
            text += "═"*60 + "\n\n"
            
            sorted_contrib = sorted(contributions.items(), key=lambda x: abs(x[1]), reverse=True)
            
            for feature, contribution in sorted_contrib:
                direction = "↑ INCREASES strength" if contribution > 0 else "↓ DECREASES strength"
                effect = abs(contribution)
                
                # Create visual bar
                bar_length = int(min(abs(effect) * 50, 20))
                bar = '█' * bar_length + '░' * (20 - bar_length)
                
                text += f"• {feature[:25]:25} {direction:24} |{bar}| {effect:+.4f} MPa\n"
                
                # Add interpretation
                if abs(contribution) > 2.0:
                    text += f"  ⚡ Very strong {'positive' if contribution > 0 else 'negative'} impact\n"
                elif abs(contribution) > 1.0:
                    text += f"  📈 Strong {'positive' if contribution > 0 else 'negative'} impact\n"
                elif abs(contribution) > 0.5:
                    text += f"  📊 Moderate {'positive' if contribution > 0 else 'negative'} impact\n"
                else:
                    text += f"  📉 Minor impact\n"
            
            self.contributions_text.insert(1.0, text)
        else:
            self.contributions_text.insert(1.0, "No contribution data available.")
        
        self.contributions_text.config(state='disabled')
    
    def update_comparison_text(self, comparisons, prediction):
        self.comparison_text.config(state='normal')
        self.comparison_text.delete(1.0, tk.END)
        
        text = f"📈 COMPARISON WITH TYPICAL VALUES\n"
        text += "═"*60 + "\n\n"
        
        for comp in comparisons:
            text += f"• {comp['feature'][:25]:25}\n"
            text += f"  Current: {comp['value']:.2f} | Typical: {comp['mean']:.2f}\n"
            diff = comp['value'] - comp['mean']
            if abs(diff) > 0.001:
                pct_diff = (diff / comp['mean']) * 100 if comp['mean'] != 0 else 0
                if diff > 0:
                    text += f"  📈 +{pct_diff:.1f}% above typical\n"
                else:
                    text += f"  📉 {pct_diff:.1f}% below typical\n"
            text += "\n"
        
        # Add interpretation
        text += "\n🔍 INTERPRETATION:\n"
        text += "═"*40 + "\n"
        
        if prediction >= 40:
            text += "• Excellent compressive strength - Suitable for high-performance concrete\n"
            text += "• Mix design shows good characteristics for structural applications\n"
        elif prediction >= 30:
            text += "• Good compressive strength - Suitable for reinforced concrete structures\n"
            text += "• Standard mix design for building construction\n"
        elif prediction >= 20:
            text += "• Moderate compressive strength - Suitable for general construction\n"
            text += "• Adequate for slabs, foundations, and non-critical elements\n"
        else:
            text += "• Low compressive strength - Limited structural applications\n"
            text += "• Consider mix optimization for better performance\n"
        
        self.comparison_text.insert(1.0, text)
        self.comparison_text.config(state='disabled')
    
    def show_success_animation(self):
        original_bg = self.predict_btn.cget('style')
        self.predict_btn.configure(style='Success.TButton')
        self.root.after(1000, lambda: self.predict_btn.configure(style=original_bg))
    
    def save_to_history(self, prediction, classification, inputs):
        # Get top feature for summary
        top_feature = self.feature_importance.iloc[0]['feature']
        top_value = inputs.get(top_feature, 0)
        
        history_entry = {
            'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'compressive_strength': float(prediction),
            'classification': classification,
            'model': 'Random_Forest_PSO',
            'importance_type': 'Permutation',
            'top_feature': f"{top_feature}: {top_value:.2f}",
            **{k: inputs[k] for k in self.feature_importance.head(3)['feature'].tolist() if k in inputs}
        }
        
        self.prediction_history.append(history_entry)
        self.update_history_display()
        self.save_history()
    
    def update_history_display(self):
        # Clear existing items
        for item in self.history_tree.get_children():
            self.history_tree.delete(item)
        
        # Add history entries
        for entry in reversed(self.prediction_history[-100:]):  # Last 100 entries
            self.history_tree.insert('', 'end', values=(
                entry['timestamp'],
                f"{entry['compressive_strength']:.2f}",
                entry['classification'],
                entry.get('top_feature', 'N/A')
            ))
        
        if self.prediction_history:
            times = [h['compressive_strength'] for h in self.prediction_history]
            stats_text = (f"📊 Total Predictions: {len(self.prediction_history)} | "
                         f"Average: {np.mean(times):.2f} MPa | "
                         f"Range: {min(times):.2f} - {max(times):.2f} MPa")
            self.history_stats_var.set(stats_text)
    
    def load_history_entry(self, event):
        selection = self.history_tree.selection()
        if selection:
            item = self.history_tree.item(selection[0])
            values = item['values']
            
            # Find corresponding history entry
            timestamp = values[0]
            for entry in self.prediction_history:
                if entry['timestamp'] == timestamp:
                    # Fill inputs with historical values
                    for feature in self.entries:
                        if feature in entry:
                            self.entries[feature].delete(0, tk.END)
                            self.entries[feature].insert(0, f"{entry[feature]:.2f}")
                    
                    # Update result display
                    self.result_var.set(f"{entry['compressive_strength']:.2f}")
                    self.classification_var.set(entry['classification'])
                    
                    messagebox.showinfo("History Loaded", 
                                       f"✅ Loaded prediction from {timestamp}")
                    break
    
    def clear_inputs(self):
        for feature, entry in self.entries.items():
            entry.delete(0, tk.END)
            entry.insert(0, f"{self.feature_stats[feature]['mean']:.2f}")
        
        self.result_var.set("---")
        self.classification_var.set("Not predicted")
        
        self.contributions_text.config(state='normal')
        self.contributions_text.delete(1.0, tk.END)
        self.contributions_text.insert(1.0, "Feature contributions will appear here after prediction.")
        self.contributions_text.config(state='disabled')
        
        self.comparison_text.config(state='normal')
        self.comparison_text.delete(1.0, tk.END)
        self.comparison_text.insert(1.0, "Comparison with typical values will appear here.")
        self.comparison_text.config(state='disabled')
    
    def save_result(self):
        """Save current prediction result - NO STATISTICAL PARAMETERS AND NO CONFIDENCE"""
        result_text = f"""═══════════════════════════════════════════════════════════════
   COMPRESSIVE STRENGTH PREDICTION - RANDOM FOREST + PSO
═══════════════════════════════════════════════════════════════

Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Model: Random Forest Regressor (PSO-Optimized)
Importance: Permutation Feature Importance
Train-Test Split: 70-30

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
                    PREDICTION RESULT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Predicted Compressive Strength: {self.result_var.get()} MPa
Classification: {self.classification_var.get()}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
         PSO-OPTIMIZED HYPERPARAMETERS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n"""
        
        for param, value in self.hyperparams.items():
            result_text += f"  • {param:25}: {value}\n"
        
        result_text += "\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
        result_text += "                    INPUT PARAMETERS\n"
        result_text += "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n"
        
        for feature in self.entries:
            result_text += f"{feature:30}: {self.entries[feature].get()}\n"
        
        filename = filedialog.asksaveasfilename(
            defaultextension=".txt",
            filetypes=[("Text files", "*.txt"), ("All files", "*.*")],
            initialfile=f"strength_prediction_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
        )
        
        if filename:
            try:
                with open(filename, 'w') as f:
                    f.write(result_text)
                messagebox.showinfo("Save Successful", f"✅ Result saved to {filename}")
            except Exception as e:
                messagebox.showerror("Save Failed", f"❌ Failed to save: {str(e)}")
    
    def copy_to_clipboard(self):
        """Copy result to clipboard"""
        result_text = f"Compressive Strength: {self.result_var.get()} MPa ({self.classification_var.get()}) - Random Forest + PSO Model"
        self.root.clipboard_clear()
        self.root.clipboard_append(result_text)
        messagebox.showinfo("Copied", "✅ Result copied to clipboard!")
    
    def export_results(self):
        """Export detailed results"""
        self.export_history()
    
    def clear_history(self):
        if not self.prediction_history:
            messagebox.showinfo("No History", "📭 Prediction history is already empty.")
            return
        
        if messagebox.askyesno("Clear History", "🗑️ Clear all prediction history?"):
            self.prediction_history = []
            self.update_history_display()
            self.save_history()
            messagebox.showinfo("History Cleared", "✅ Prediction history cleared.")
    
    def export_history(self):
        if not self.prediction_history:
            messagebox.showwarning("No Data", "⚠️ No prediction history to export.")
            return
        
        filename = filedialog.asksaveasfilename(
            defaultextension=".csv",
            filetypes=[("CSV files", "*.csv"), ("Excel files", "*.xlsx"), ("All files", "*.*")],
            initialfile=f"strength_predictions_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        )
        
        if filename:
            try:
                df = pd.DataFrame(self.prediction_history)
                if filename.endswith('.xlsx'):
                    df.to_excel(filename, index=False)
                else:
                    df.to_csv(filename, index=False)
                messagebox.showinfo("Export Successful", f"✅ History exported to {filename}")
            except Exception as e:
                messagebox.showerror("Export Failed", f"❌ Failed to export: {str(e)}")
    
    def load_history_from_file(self):
        filename = filedialog.askopenfilename(
            filetypes=[("JSON files", "*.json"), ("CSV files", "*.csv"), ("All files", "*.*")]
        )
        
        if filename:
            try:
                if filename.endswith('.json'):
                    with open(filename, 'r') as f:
                        self.prediction_history = json.load(f)
                elif filename.endswith('.csv'):
                    self.prediction_history = pd.read_csv(filename).to_dict('records')
                
                self.update_history_display()
                messagebox.showinfo("Load Successful", 
                                   f"✅ Loaded {len(self.prediction_history)} predictions")
            except Exception as e:
                messagebox.showerror("Load Failed", f"❌ Failed to load: {str(e)}")
    
    def save_history(self):
        try:
            with open(self.history_file, 'w') as f:
                json.dump(self.prediction_history, f, indent=2)
        except Exception as e:
            print(f"Failed to save history: {e}")
    
    def load_history(self):
        try:
            if os.path.exists(self.history_file):
                with open(self.history_file, 'r') as f:
                    self.prediction_history = json.load(f)
                self.update_history_display()
        except:
            pass
    
    # ========== PLOTTING FUNCTIONS ==========
    
    def save_plot(self, plot_function, filename_suffix):
        """Helper function to save individual plots"""
        fig = plot_function(save_mode=True)
        if fig:
            filename = os.path.join(self.save_dir, f"{filename_suffix}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png")
            fig.savefig(filename, dpi=300, bbox_inches='tight')
            print(f"✅ Saved: {filename}")
            plt.close(fig)
            return True
        return False
    
    def save_all_plots(self):
        """Save all plots to the specified folder"""
        try:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            saved_plots = []
            
            # Create subfolder for this batch
            batch_folder = os.path.join(self.save_dir, f"plots_{timestamp}")
            os.makedirs(batch_folder, exist_ok=True)
            original_save_dir = self.save_dir
            self.save_dir = batch_folder
            
            plot_functions = [
                (self.plot_feature_importance, "permutation_importance"),
                (self.plot_data_distribution, "data_distribution"),
                (self.plot_correlation_matrix, "correlation_matrix"),
                (self.plot_feature_statistics, "feature_statistics"),
                (self.plot_mix_parameters, "mix_parameters"),
            ]
            
            for func, name in plot_functions:
                try:
                    if self.save_plot(func, name):
                        saved_plots.append(name)
                except Exception as e:
                    print(f"Failed to save {name}: {e}")
            
            self.save_dir = original_save_dir
            
            if saved_plots:
                messagebox.showinfo("Plots Saved", 
                                   f"✅ Successfully saved {len(saved_plots)} plots to:\n{batch_folder}")
            else:
                messagebox.showwarning("No Plots", "⚠️ No plots were generated to save.")
                
        except Exception as e:
            messagebox.showerror("Save Error", f"❌ Failed to save plots: {str(e)}")
    
    def plot_feature_importance(self, save_mode=False):
        if not save_mode:
            self.clear_analysis_frame()
        
        fig = Figure(figsize=(12, 8), dpi=100, facecolor='white')
        ax = fig.add_subplot(111)
        
        # Plot top 15 features with error bars
        importance_df = self.feature_importance.head(15)
        y_pos = np.arange(len(importance_df))
        
        # Create color gradient
        colors = cm.Greens(np.linspace(0.3, 0.9, len(importance_df)))
        
        bars = ax.barh(y_pos, importance_df['importance'], 
                      color=colors, edgecolor=self.colors['rf_green'], 
                      height=0.7, linewidth=1,
                      xerr=importance_df['std'], capsize=3)
        
        ax.set_yticks(y_pos)
        ax.set_yticklabels(importance_df['feature'], fontsize=10)
        ax.set_xlabel('Permutation Importance Score', fontsize=12, fontweight='bold')
        ax.set_title(f'Top 15 Features - Permutation Importance\n{self.target_name} - Random Forest + PSO', 
                    fontsize=14, fontweight='bold', pad=20, color=self.colors['rf_green'])
        ax.grid(True, alpha=0.3, axis='x', linestyle='--')
        
        # Add value labels
        total_importance = importance_df['importance'].sum()
        for i, (bar, importance, std) in enumerate(zip(bars, importance_df['importance'], importance_df['std'])):
            percentage = (importance / total_importance) * 100
            ax.text(bar.get_width() + std + 0.001, bar.get_y() + bar.get_height()/2,
                   f'{importance:.4f} ± {std:.4f}\n({percentage:.1f}%)', 
                   va='center', fontsize=8, fontweight='bold')
        
        fig.tight_layout()
        
        if not save_mode:
            canvas = FigureCanvasTkAgg(fig, self.analysis_frame)
            canvas.draw()
            widget = canvas.get_tk_widget()
            widget.pack(fill='both', expand=True)
            self.current_plot = fig
        
        return fig
    
    def plot_data_distribution(self, save_mode=False):
        if not save_mode:
            self.clear_analysis_frame()
        
        fig = Figure(figsize=(12, 8), dpi=100, facecolor='white')
        
        # Create subplot
        ax = fig.add_subplot(111)
        
        # Create histogram
        n, bins, patches = ax.hist(self.y, bins=30, alpha=0.7, 
                                  color=self.colors['sandy'], 
                                  edgecolor=self.colors['soil_brown'], 
                                  density=True, label='Distribution')
        
        ax.set_xlabel(f'{self.target_name} (MPa)', fontsize=12, fontweight='bold')
        ax.set_ylabel('Density', fontsize=12, fontweight='bold')
        ax.set_title(f'Distribution of {self.target_name}', 
                    fontsize=14, fontweight='bold', pad=20, color=self.colors['soil_brown'])
        ax.grid(True, alpha=0.3, linestyle='--')
        
        fig.tight_layout()
        
        if not save_mode:
            canvas = FigureCanvasTkAgg(fig, self.analysis_frame)
            canvas.draw()
            widget = canvas.get_tk_widget()
            widget.pack(fill='both', expand=True)
            self.current_plot = fig
        
        return fig
    
    def plot_correlation_matrix(self, save_mode=False):
        if not save_mode:
            self.clear_analysis_frame()
        
        fig = Figure(figsize=(14, 12), dpi=100, facecolor='white')
        ax = fig.add_subplot(111)
        
        # Calculate correlation for top features
        top_features = self.feature_importance.head(10)['feature'].tolist()
        if self.target_name in self.df.columns:
            corr_features = top_features + [self.target_name]
            corr_matrix = self.df[corr_features].corr()
            
            im = ax.imshow(corr_matrix, cmap='Greens', vmin=-1, vmax=1, aspect='auto')
            
            ax.set_xticks(np.arange(len(corr_features)))
            ax.set_yticks(np.arange(len(corr_features)))
            ax.set_xticklabels([f[:10] for f in corr_features], rotation=45, ha='right', fontsize=9)
            ax.set_yticklabels([f[:10] for f in corr_features], fontsize=9)
            
            ax.set_title(f'Correlation Matrix', 
                        fontsize=14, fontweight='bold', pad=20, color=self.colors['rf_green'])
            
            # Add correlation values
            for i in range(len(corr_features)):
                for j in range(len(corr_features)):
                    text = ax.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}',
                                 ha="center", va="center", color="black", fontsize=8)
            
            cbar = ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            cbar.ax.set_ylabel('Correlation Coefficient', rotation=-90, va="bottom", fontsize=10)
        
        fig.tight_layout()
        
        if not save_mode:
            canvas = FigureCanvasTkAgg(fig, self.analysis_frame)
            canvas.draw()
            widget = canvas.get_tk_widget()
            widget.pack(fill='both', expand=True)
            self.current_plot = fig
        
        return fig
    
    def plot_feature_statistics(self, save_mode=False):
        if not save_mode:
            self.clear_analysis_frame()
        
        fig = Figure(figsize=(14, 10), dpi=100, facecolor='white')
        
        # Plot statistics for top 6 features
        top_features = self.feature_importance.head(6)['feature'].tolist()
        
        for i, feature in enumerate(top_features, 1):
            ax = fig.add_subplot(2, 3, i)
            
            data = self.X[feature].dropna()
            
            # Box plot
            bp = ax.boxplot(data, vert=True, patch_artist=True, 
                           widths=0.6, showfliers=True)
            
            bp['boxes'][0].set_facecolor(self.colors['sandy'])
            bp['boxes'][0].set_alpha(0.7)
            bp['boxes'][0].set_edgecolor(self.colors['rf_green'])
            bp['boxes'][0].set_linewidth(2)
            
            bp['whiskers'][0].set_color(self.colors['rf_green'])
            bp['whiskers'][1].set_color(self.colors['rf_green'])
            bp['caps'][0].set_color(self.colors['rf_green'])
            bp['caps'][1].set_color(self.colors['rf_green'])
            bp['medians'][0].set_color(self.colors['danger'])
            
            ax.set_ylabel('Value', fontsize=9, fontweight='bold')
            ax.set_title(f'{feature[:15]}', fontsize=10, fontweight='bold')
            ax.grid(True, alpha=0.3, axis='y', linestyle='--')
        
        fig.suptitle(f'Statistical Distribution of Top 6 Features', 
                    fontsize=16, fontweight='bold', y=0.98, color=self.colors['rf_green'])
        fig.tight_layout(rect=[0, 0.03, 1, 0.95])
        
        if not save_mode:
            canvas = FigureCanvasTkAgg(fig, self.analysis_frame)
            canvas.draw()
            widget = canvas.get_tk_widget()
            widget.pack(fill='both', expand=True)
            self.current_plot = fig
        
        return fig
    
    def plot_mix_parameters(self, save_mode=False):
        """Additional plot for mix parameters visualization"""
        if not save_mode:
            self.clear_analysis_frame()
        
        fig = Figure(figsize=(14, 10), dpi=100, facecolor='white')
        
        # Select top features for mix parameters
        mix_features = self.feature_importance.head(6)['feature'].tolist()
        
        for i, feature in enumerate(mix_features, 1):
            ax = fig.add_subplot(2, 3, i)
            
            # Scatter plot with target
            ax.scatter(self.X[feature], self.y, 
                      alpha=0.5, c=self.colors['sandy'], 
                      edgecolors=self.colors['rf_green'])
            
            # Add trend line
            z = np.polyfit(self.X[feature], self.y, 1)
            p = np.poly1d(z)
            x_range = np.linspace(self.X[feature].min(), self.X[feature].max(), 100)
            ax.plot(x_range, p(x_range), "r--", alpha=0.8, linewidth=2)
            
            ax.set_xlabel(feature[:15], fontsize=9, fontweight='bold')
            ax.set_ylabel('Compressive Strength (MPa)', fontsize=9, fontweight='bold')
            ax.set_title(f'{feature[:15]} vs Strength', fontsize=10, fontweight='bold')
            ax.grid(True, alpha=0.3, linestyle='--')
        
        fig.suptitle('Mix Parameters vs Compressive Strength', 
                    fontsize=16, fontweight='bold', y=0.98, color=self.colors['rf_green'])
        fig.tight_layout(rect=[0, 0.03, 1, 0.95])
        
        if not save_mode:
            canvas = FigureCanvasTkAgg(fig, self.analysis_frame)
            canvas.draw()
            widget = canvas.get_tk_widget()
            widget.pack(fill='both', expand=True)
            self.current_plot = fig
        
        return fig
    
    def clear_analysis_frame(self):
        for widget in self.analysis_frame.winfo_children():
            widget.destroy()
    
    def run(self):
        print("\n" + "="*70)
        print("🏗️ COMPRESSIVE STRENGTH PREDICTOR - RANDOM FOREST + PSO")
        print("="*70)
        print(f"📊 Target Variable: {self.target_name}")
        print(f"🔢 Number of Features: {len(self.feature_names)}")
        print(f"📈 Data Split: 70% Training, 30% Testing")
        print(f"🚀 Algorithm: Random Forest Regressor with PSO-optimized hyperparameters")
        print(f"📊 Importance: Permutation Feature Importance")
        print(f"💾 Plot save directory: {self.save_dir}")
        print("="*70)
        print("✅ Application ready! Starting GUI...")
        print("🏁 Press Ctrl+C in terminal to stop\n")
        
        # Auto-load a preset
        self.root.after(100, lambda: self.load_preset_by_name("Medium Strength Mix"))
        
        self.root.mainloop()

# Run the application
if __name__ == "__main__":
    try:
        print("🏗️ Initializing Compressive Strength Predictor...")
        app = CompressiveStrengthPredictor()
        app.run()
    except KeyboardInterrupt:
        print("\n👋 Application terminated by user")
    except Exception as e:
        print(f"❌ Application Error: {str(e)}")
        import traceback
        traceback.print_exc()
        messagebox.showerror("Application Error", f"Failed to start:\n\n{str(e)}")

🏗️ Initializing Compressive Strength Predictor...
📂 Loading data from: D:\2026 Work\My Papers\2-Compressive Strength\DATA\Zero Normilisation.csv
✅ Dataset loaded successfully!
📐 Dataset shape: (198, 12)
🔧 Cleaned columns: ['C', 'CA', 'FA', 'W', 'blast_furnace_slag', 'fly_ash', 'W_B', 'CCU_steam_curing_time', 'T', 'CC', 'P', 'CS']
✅ Target column found: 'CS'
🎯 Target variable: 'CS'
🔍 Features loaded: 11
📊 Features: C, CA, FA, W, blast_furnace_slag, fly_ash, W_B, CCU_steam_curing_time, T, CC...

📊 Target Statistics - CS:
   • Mean: 32.23 MPa
   • Std: 18.84 MPa
   • Min: 1.61 MPa
   • Max: 81.89 MPa

📊 TRAIN-TEST SPLIT (70-30)

📈 Training set size: 138 samples (69.7%)
📉 Testing set size: 60 samples (30.3%)

🚀 TRAINING RANDOM FOREST MODEL WITH PSO OPTIMIZED HYPERPARAMETERS

📋 PSO-Optimized Hyperparameters:
   • n_estimators             : 265
   • max_depth                : 10
   • min_samples_split        : 2
   • min_samples_leaf         : 1
   • max_features             : 0.565
   • max